In [1]:
!pip install --no-cache-dir -r requirements.txt
!pip install --no-cache-dir tqdm

In [2]:
import os
import sys

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

import faiss

sys.path.append('../common/')
from utils import get_embedding, load_pdf, split_document 

In [3]:
VS_DB_DIR = "vector_db"
PERSISTENT_VECTOR_STORE_NAME = f"{VS_DB_DIR}/FAISS_vector_store"
PAPERS_DIR = "papers/"

# number of workers to batch process documents
N_WORKERS = 4

In [4]:
# Set up vector store
embeddings = get_embedding("main_embedding_model")
embedding_dim = len(embeddings.embed_query("Not with a bang but a whimper."))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [5]:
from uuid import uuid4

def flatten_texts_n_metadatas(main_list: list[tuple]) -> tuple[list, list]:
    return ([item for sublists in main_list for item in sublists[0]],
            [item for sublists in main_list for item in sublists[1]])

def load_file(file_n_area: tuple) -> tuple:
    file_name, area = file_n_area
    title, content, metadata  = load_pdf(file_name)
    
    return {"title": title,
            "area": area,
            "content": content,
            "metadata": metadata}

def save_document(paper: tuple, save_dir: str) -> None:
    file_name = str(uuid4())
    with open(f"{save_dir}/{file_name}.md", "w") as out_file:
        out_file.write(paper["content"])
    
    paper["doc_id"] = file_name


def get_texts_n_metadatas(paper: tuple) -> tuple[list, list]:
    texts, metadatas = [], []    
    chunk, centers = split_document(paper["content"])
    for i, (chunk, center) in enumerate(zip(chunk, centers)):
        texts.append(chunk)
        metadatas.append({
            "title": paper["title"],
            "center": center,
            "chunk": i,
            "area": paper["area"],
            "doc_id": paper["doc_id"],
            "extra_metadata": paper["metadata"]
        })
    
    return texts, metadatas

#  
def process_document(file_n_area: tuple,
                     save_dir: str = f"{VS_DB_DIR}/documents"
                    ) -> tuple[list, list]:
    """
    Loads, saves in db and slipts document.
    Returns a tuple (d=2) of lists with texts and metadatas
    """
    paper = load_file(file_n_area)
    save_document(paper, save_dir)
    return get_texts_n_metadatas(paper)

In [6]:
from tqdm import tqdm
from multiprocessing import Pool

papers_files = []
for area in os.listdir(PAPERS_DIR):
    for paper in os.listdir(f"{PAPERS_DIR}{area}"):
        papers_files.append((f"{PAPERS_DIR}{area}/{paper}", area))

# very lazily process files in parallel
with Pool(N_WORKERS) as p, tqdm(total=len(papers_files)) as progress_bar:
    for texts, metadatas in p.imap(process_document, papers_files):
        vector_store.add_texts(texts=texts,
                               metadatas=metadatas)
        progress_bar.update()

  0%|          | 0/12 [00:00<?, ?it/s]

OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
O

  8%|▊         | 1/12 [00:21<03:58, 21.67s/it]

OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
O

 17%|█▋        | 2/12 [00:42<03:34, 21.46s/it]

OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
OpenCV not installed. Disabling OCR.
O

100%|██████████| 12/12 [02:27<00:00, 12.26s/it]


In [7]:
vector_store.save_local(PERSISTENT_VECTOR_STORE_NAME)

In [8]:
for metadata in metadatas:
    print(metadata)

{'title': '**A FASTER HEURISTIC FOR THE TRAVELING SALESMAN PROBLEM WITH DRONE**', 'center': 1395, 'chunk': 0, 'area': 'CS', 'doc_id': '0323897d-caf9-482f-88ec-9fdc799889d7', 'extra_metadata': {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-05-30T00:08:23+00:00', 'source': 'papers/CS/A FASTER HEURISTIC FOR THE TRAVELING SALESMAN PROBLEM WITH DRONE.pdf', 'file_path': 'papers/CS/A FASTER HEURISTIC FOR THE TRAVELING SALESMAN PROBLEM WITH DRONE.pdf', 'total_pages': 17, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-05-30T00:08:23+00:00', 'trapped': '', 'modDate': 'D:20240530000823Z', 'creationDate': 'D:20240530000823Z'}}
{'title': '**A FASTER HEURISTIC FOR THE TRAVELING SALESMAN PROBLEM WITH DRONE**', 'center': 4110, 'chunk': 1, 'area': 'CS', 'doc_id': '0323897d-caf9-482f-88ec-9fdc799889d7', 'extra_metadata': {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-05-30T00: